In [ ]:
import os
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.preprocessing.image import ImageDataGenerator


In [ ]:
# 1. CONFIGURATION

DATASET_ROOT = "Output/segmented_images/purple_bg" 
CSV_NAME = "labelled_dataset.csv"
IMG_SIZE = (224, 224)
BATCH_SIZE = 4 
EPOCHS = 100

# 2. CREATE OR LOAD CSV

if not os.path.exists(CSV_NAME):
    print(f"Scanning {DATASET_ROOT} to create {CSV_NAME}...")
    data_rows = []
    for label in ["Normal", "Abnormal"]:
        label_path = os.path.join(DATASET_ROOT, label)
        if not os.path.exists(label_path): continue
        for breed in os.listdir(label_path):
            breed_path = os.path.join(label_path, breed)
            if not os.path.isdir(breed_path): continue
            for img in os.listdir(breed_path):
                if img.lower().endswith((".jpg", ".jpeg", ".png")):
                    data_rows.append({
                        "file_path": os.path.join(breed_path, img), 
                        "label": label, 
                        "breed": breed
                    })
    df = pd.DataFrame(data_rows)
    df.to_csv(CSV_NAME, index=False)
    print(f"Created {CSV_NAME} with {len(df)} entries.")
else:
    df = pd.read_csv(CSV_NAME)
    if 'filepath' in df.columns:
        df = df.rename(columns={"filepath": "file_path"})

# Label Encoding
if 'label_encoded' not in df.columns:
    df['label_encoded'] = df['label'].map({'Normal': 0, 'Abnormal': 1})


# 3. THE 3-WAY SPLIT (Train, Validation, Test)

# A. Isolate 15% for the Final Test (Never seen during training)
trainval_df, test_df = train_test_split(
    df, test_size=0.15, stratify=df['label_encoded'], random_state=42
)

# B. Split the remaining 85% into Train and Validation
train_df, val_df = train_test_split(
    trainval_df, test_size=0.20, stratify=trainval_df['label_encoded'], random_state=42
)

print("\nData Split Complete:")
print(f"  Training:   {len(train_df)} images")
print(f"  Validation: {len(val_df)} images (Used for Early Stopping)")
print(f"  Testing:    {len(test_df)} images (The Final Exam)")

In [ ]:
# 4. GENERATORS & AUGMENTATION

# Augmentation ONLY on the training data
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

# Validation and Test get STRICTLY clean data
clean_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

train_gen = train_datagen.flow_from_dataframe(
    train_df, x_col='file_path', y_col='label_encoded',
    target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='raw', shuffle=True
)

val_gen = clean_datagen.flow_from_dataframe(
    val_df, x_col='file_path', y_col='label_encoded',
    target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='raw', shuffle=False
)

test_gen = clean_datagen.flow_from_dataframe(
    test_df, x_col='file_path', y_col='label_encoded',
    target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='raw', shuffle=False
)

# Class Weights
weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_df['label_encoded']),

    y=train_df['label_encoded']
)
class_weights = {0: weights[0], 1: weights[1] * 1.3}


In [ ]:
# 5. BUILD MODEL ARCHITECTURE

print("\nBuilding Final Model...")
tf.keras.backend.clear_session()
base_model = MobileNetV2(input_shape=(*IMG_SIZE, 3), include_top=False, weights='imagenet')
base_model.trainable = False 

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dropout(0.5)(x)
predictions = Dense(1, activation='sigmoid')(x)
model = Model(inputs=base_model.input, outputs=predictions)


# 6. STAGE 1: WARMUP

print("\n--- STAGE 1: Warmup (Head Only) ---")
model.compile(optimizer=Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=['accuracy'])
model.fit(train_gen, epochs=15, validation_data=val_gen, class_weight=class_weights, verbose=1)


In [ ]:
# 7. STAGE 2: FINE-TUNING

print("\n--- STAGE 2: Fine-Tuning (Full Model) ---")
base_model.trainable = True
for layer in base_model.layers: 
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False
        
model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.Precision(name='precision'), tf.keras.metrics.Recall(name='recall')]
)

callbacks = [
    EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, min_lr=1e-7)
]

history = model.fit(
    train_gen, validation_data=val_gen, epochs=EPOCHS,
    callbacks=callbacks, class_weight=class_weights, verbose=1
)


In [ ]:
# 8. TEST SET EVALUATION

print("\n" + "="*50)
print("FINAL TEST SET REPORT CARD")
print("="*50)
test_loss, test_acc, test_prec, test_rec = model.evaluate(test_gen, verbose=0)

print(f"Test Accuracy:  {test_acc*100:.2f}%")
print(f"Test Precision: {test_prec:.4f}")
print(f"Test Recall:    {test_rec:.4f}")


In [ ]:
# Save the final model
model.save("Final_Dog_Posture_Model.keras")
print("\n✓ Model successfully saved as 'Final_Dog_Posture_Model.keras'")